The sole purpose of this file is to convert the CombinedCSI.mat file from the pipeline to a separate data.npy and mask.npy files for easier usage. You only have to insert the correct data folder name below and run the notebook. A CombinedCSI.mat file in that folder is required

In [ ]:
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import sys
import os

%matplotlib widget

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from denoising.config.load import load_yaml
from denoising.config.build import build_config
from denoising.inference.api import infer
from src.denoising.figures.EvalBeforeFitting import *
from denoising.data.data_utils import *

In [ ]:
##### Set LW Parameter

SNR = 1

In [ ]:
data_folder_name = f"/workspace/Denoising/datasets/DMI/Simulations/Lesion_double_radius/Simulated_Lesion_GT_double"   #Simulated_Lesion_GT  Simulated_Lesion_GT_double

data = loadmat(f'{data_folder_name}/Lesion_120pts_DoubleRadius.mat')

Daten = data['csi_data_lr']['Data'][0,0]

MAX = np.abs(np.max(Daten))  

Daten_fft = np.fft.ifftshift(np.fft.fft(np.fft.fftshift(Daten, axes=-2), axis=-2), axes=-2)

In [ ]:
def Sigma(SNR):
    sigma = 878.9994/SNR
    return sigma

# Simulate noise kspace

In [ ]:
def Sigma(SNR):
    sigma = 878.9994/SNR
    return sigma

for i in range(1,7):

    outpath = f"/workspace/Denoising/datasets/DMI/Simulations/SNRExperiments_double_radius/SNR{SNR}/Noise_{i}"

    noise_data = add_kspace_noise(
        Daten,
        noise_sigma=Sigma(SNR),#97.6666 SNR_9 = 97.6666 (main paper!), 
        seed=i,
        apply_hamming=True,
        #save_path = outpath
    )

noise_data_fft = np.fft.ifftshift(np.fft.fft(np.fft.fftshift(noise_data, axes=-2), axis=-2), axes=-2)

# Visualize spectra

In [ ]:
noiseless = np.load(data_folder_name+"/data.npy")

noiseless_fft = np.fft.ifftshift(np.fft.fft(np.fft.fftshift(noiseless, axes=-2), axis=-2), axes=-2)

In [ ]:
T = -1

data1 = np.abs(noiseless_fft[...,T])

data2 = np.abs(noise_data_fft[...,T])

image_volume = np.abs(Daten[...,1,-1])   # shape (X, Y, Z)

interactive_spectra_viewer(
    image_volume=image_volume,
    spectrum_func1=lambda x, y, z: data1[x, y, z, :],
    spectrum_func2=lambda x, y, z: data2[x, y, z, :],
    label1="GT",
    label2="Noisy",
    f_min=0,
    #f_max=60,
    cmap="plasma",
)